# **Análisis de las reseñas (nuevos datos)**
El objetivo de este notebook es analizar los nuevos datos del problema de reseñas extraidos en la cuarta fase del proyecto.

---
## Importación de librerías

In [ ]:
# Módulo
import re
import plotly.graph_objects as go
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import nltk
import numpy as np
from pandas import Series
import plotly.graph_objects as go
from nltk.corpus import stopwords
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from src.utils.config import load_env_file
from src.utils.config import seed
from textblob import TextBlob
from plotly.subplots import make_subplots


load_env_file()

# Descarga usando nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
from src.utils.files import read_file
from src.utils.config import reviews, new_data_reviews

## Carga de datos
Configuración del Minio, poner en True para usar la información de Minio

In [ ]:
use_minio = True # Solo cambiar este parámetro
minio = {"minio_write": False, "minio_read": use_minio}

In [ ]:
df_datos = read_file(reviews, minio)
df_nuevo = read_file(new_data_reviews, minio)

In [ ]:
counts_datos = df_datos["is_positive"].value_counts().sort_index()
count_nuevo = df_nuevo["is_positive"].value_counts().sort_index()

fig = make_subplots(rows=1,cols=2, subplot_titles= ("Datos existentes","Nuevos datos extraídos"))

fig.add_trace(go.Bar(
        x=counts_datos.index.astype(str),
        y=counts_datos.values,
        text=counts_datos.values,
        textposition='inside'
), row = 1, col=1)

fig.add_trace(go.Bar(
    x=count_nuevo.index.astype(str),
    y=count_nuevo.values,
    text=count_nuevo.values,
    textposition='inside'
), row = 1, col = 2)

fig.update_layout(
    xaxis_title='is_positive',
    yaxis_title='Frecuencia',
    xaxis_title_font=dict(family='Times New Roman'),
    yaxis_title_font=dict(family='Times New Roman'),
    title={
        'text': 'Distribución de is_positive',
        'x': 0.5,
        'font': dict(size=30, weight="bold", family='Times New Roman')
    },
    showlegend = False
)

fig.update_annotations(font=dict(family="Times New Roman"))
fig.show()

In [ ]:
print(f"Ratio de mensajes negativos sobre positivos en los datos existentes: {len(df_datos[~df_datos["is_positive"]])/len(df_datos[df_datos["is_positive"]]):.3f}")
print(f"Ratio de mensajes negativos sobre positivos en los datos nuevos: {len(df_nuevo[~df_nuevo["is_positive"]])/len(df_nuevo["is_positive"]):.3f}")

No se aprecia un gran cambio en la distribución de las reseñas positivas y negativas, aunque la proporción de negativos parece disminuir de 33% a 19%. Probablemente se deba a la falta de nuevos juegos relevantes, que son los que más reseñas negativas tienen. 

In [ ]:
length_datos = df_datos["text"].astype(str).apply(len)
length_nuevo = df_nuevo["text"].astype(str).apply(len)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Datos existentes", "Nuevos datos extraídos"))

fig.add_trace(go.Histogram(
    x=length_datos
), row=1, col=1)

fig.add_trace(go.Histogram(
    x=length_nuevo
), row=1, col=2)

fig.update_layout(
    title={
        'text': 'Distribución de Longitud de Comentarios',
        'x': 0.5,
        'font': dict(size=30, weight="bold", family='Times New Roman')
    },
    showlegend=False
)

fig.update_annotations(font=dict(family="Times New Roman"))
fig.show()

La distribución de longitudes de comentarios es la misma.

In [ ]:
import re
from nltk.corpus import stopwords
from wordcloud import WordCloud
import matplotlib.pyplot as plt

stop_words = set(stopwords.words("english"))
def clean_text(text, stop_words):
    """Se queda solo lo que es texto, quitando stopwords (determinantes, artículos...)"""
    text = text.lower().split()
    text = " ".join(word for word in text if word not in stop_words)
    text = re.sub(r"[^a-z\s]", "", text)
    text = text.split()
    text = " ".join(text)
    return text

def graph_wordcloud(text, stop_words, is_positive):
    """grafica una nube de palabras dado un string, utiliza un esquema de colores distinto dependiento del tipo de reseña"""
    colormaps = "summer" if is_positive else "autumn"
    wordcloud = WordCloud(width=800, height=400, max_words=100, collocations=True, stopwords=stop_words, background_color="white", min_font_size=10, colormap=colormaps, random_state=42).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")  
    plt.show()

In [ ]:
pos_text_datos = " ".join(df_datos[df_datos["is_positive"]]["text"].astype(str).tolist())
pos_text_datos = clean_text(pos_text_datos, stop_words)

neg_text_datos = " ".join(df_datos[~df_datos["is_positive"]]["text"].astype(str).tolist())
neg_text_datos = clean_text(neg_text_datos, stop_words)

pos_text_nuevo = " ".join(df_nuevo[df_nuevo["is_positive"]]["text"].astype(str).tolist())
pos_text_nuevo = clean_text(pos_text_nuevo, stop_words)

neg_text_nuevo = " ".join(df_nuevo[~df_nuevo["is_positive"]]["text"].astype(str).tolist())
neg_text_nuevo = clean_text(neg_text_nuevo, stop_words)

In [ ]:
print("Mapas de palabras para Datos Existentes")
graph_wordcloud(pos_text_datos, stop_words, True)
print("Mapas de palabras para Datos Nuevos")
graph_wordcloud(pos_text_nuevo, stop_words, True)


Aunque palabras las palabras "even" y "one" siguen apareciendo en las nuevas palabras, vemos más palabras importantes que no aparecían antes tanto como "game", "time" o "fun". Aun así esto también se debe por la inmensamente menor cantidad de comentarios que estamos analizando esta vez.

In [ ]:
print("Mapas de palabras para Datos Existentes (Negativos)")
graph_wordcloud(neg_text_datos, stop_words, False)
print("Mapas de palabras para Datos Nuevos (Negativos)")
graph_wordcloud(neg_text_nuevo, stop_words, False)

Al igual que en las palabras positivas vemos más cantidad de palabras importantes en los nuevos datos. Además de aparecer destacar palabras que antes no destacaban tanto como "developer", "nothing", "really" o "would"